# Agentic AI for Drug Discovery
### ReAct · Reflection · RAG · Multi-Agent · LangGraph — Complete Tutorial

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## What is an AI agent?

A regular LLM call: **prompt → response** (one shot).

An AI agent: **observe → think → act → observe → think → act → ...** (iterative loop with tools).

```
┌─────────────────────────────────────────────────────────────┐
│                        AGENT LOOP                           │
│                                                             │
│   Observation  →  LLM (Think)  →  Action  →  Tool Call    │
│        ↑                                         │         │
│        └─────────────── Result ─────────────────┘         │
└─────────────────────────────────────────────────────────────┘
```

In drug discovery, the agent's **tools** are cheminformatics functions:
ADMET predictor, hERG screener, PAINS filter, PubChem lookup, docking, etc.

| Section | Pattern | Key concept |
|---------|---------|-------------|
| 1. Tools | Tool layer | Functions the agent can call |
| 2. ReAct | Reason + Act | Thought → Action → Observation |
| 3. Reflexion | Self-critique | Generate → Evaluate → Revise |
| 4. RAG agent | Retrieval | Ground answers in literature |
| 5. Multi-agent | Supervisor | Specialists + coordinator |
| 6. LangGraph | State machine | Graph of agent nodes + edges |
| 7. HITL | Human-in-loop | Approve high-risk compounds |
| 8. Memory | Episodic + semantic | Don't repeat mistakes |

---
## Section 1 — The Tool Layer

Tools are ordinary Python functions. The agent decides which to call and with what arguments.

In [5]:
# ── Install ───────────────────────────────────────────────────────────────────
# !pip install langchain langchain-openai langgraph rdkit
# For Colab / OpenAI: set OPENAI_API_KEY in environment or secrets

import os, json, re, time, textwrap, warnings
from typing import Any
from dataclasses import dataclass, field
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np

print("Imports OK ✓")

Imports OK ✓


In [11]:
# ── 1.1 Define the tool registry ─────────────────────────────────────────────
# Each tool: (a) has a name the LLM uses to call it,
#            (b) has a description the LLM reads to decide when to use it,
#            (c) is a Python function that returns a string result.

@dataclass
class Tool:
    name: str
    description: str
    func: Any        # callable

    def run(self, **kwargs) -> str:
        try:
            return str(self.func(**kwargs))
        except Exception as e:
            return f"ERROR: {e}"

# ── Tool implementations ───────────────────────────────────────────────────────
def _mol(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return mol

def tool_admet(smiles: str) -> str:
    """Compute physicochemical ADMET descriptors for a molecule."""
    mol = _mol(smiles)
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    rb   = rdMolDescriptors.CalcNumRotatableBonds(mol)
    qed  = QED.qed(mol)
    ro5_viol = sum([mw>500, logp>5, hbd>5, hba>10])
    return json.dumps({
        "MW": round(mw,1), "LogP": round(logp,2), "TPSA": round(tpsa,1),
        "HBD": hbd, "HBA": hba, "RotBonds": rb,
        "QED": round(qed,3), "Ro5_violations": ro5_viol,
        "oral_bioavailability": "Likely" if ro5_viol <= 1 else "Poor"
    }, indent=2)

def tool_herg(smiles: str) -> str:
    """Predict hERG cardiotoxicity risk (IC50 estimate)."""
    mol = _mol(smiles)
    # Simplified rule-based predictor (replace with ML model in production)
    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    basic_n = sum(1 for a in mol.GetAtoms()
                  if a.GetAtomicNum() == 7 and a.GetTotalNumHs() > 0)
    ar_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    risk_score = 0.3*max(0,logp-2) + 0.002*max(0,mw-300) + 0.15*basic_n + 0.1*ar_rings
    if risk_score > 2.0:  level, ic50 = "HIGH",   f"< 1 μM  (potent blocker)"
    elif risk_score > 1.0: level, ic50 = "MEDIUM", f"1–10 μM"
    else:                  level, ic50 = "LOW",    f"> 10 μM (likely safe)"
    return json.dumps({"hERG_risk": level, "estimated_IC50": ic50,
                       "risk_score": round(risk_score,2),
                       "warning": "HIGH hERG risk: cardiac safety concern!" if level=="HIGH" else ""},
                      indent=2)
def tool_pains(smiles: str) -> str:
    """Screen for PAINS (pan-assay interference compounds) alerts."""
    mol = _mol(smiles)
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog.FilterCatalog(params)
    entry = catalog.GetFirstMatch(mol)

    if entry:
        return json.dumps({
            "pains_alert": True,
            "pattern": entry.GetDescription(),
            "recommendation": "Remove from library — likely false positive in assays"
        }, indent=2)

    return json.dumps({
        "pains_alert": False,
        "pattern": None,
        "recommendation": "No PAINS alert detected"
    }, indent=2)


def tool_similarity(smiles1: str, smiles2: str) -> str:
    """Compute Tanimoto similarity (ECFP4) between two molecules."""
    mol1, mol2 = _mol(smiles1), _mol(smiles2)
    from rdkit import DataStructs

    fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, 2, 2048)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, 2, 2048)
    tc = DataStructs.TanimotoSimilarity(fp1, fp2)

    interpretation = (
        "identical" if tc > 0.99
        else "very similar" if tc > 0.85
        else "moderately similar" if tc > 0.60
        else "different"
    )

    return json.dumps({
        "tanimoto_similarity": round(tc, 3),
        "interpretation": interpretation
    }, indent=2)


def tool_scaffold(smiles: str) -> str:
    """Extract Murcko scaffold and compute scaffold-level properties."""
    mol = _mol(smiles)
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    return json.dumps({
        "scaffold_smiles": Chem.MolToSmiles(scaffold),
        "scaffold_mw": round(Descriptors.MolWt(scaffold), 1),
        "aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(scaffold),
        "ring_count": rdMolDescriptors.CalcNumRings(scaffold)
    }, indent=2)


def tool_lookup_pubchem(name: str) -> str:
    """Look up a compound by name in PubChem (simulated)."""
    DB = {
        "aspirin":     {"CID": 2244,  "SMILES": "CC(=O)Oc1ccccc1C(=O)O", "MW": 180.16},
        "ibuprofen":   {"CID": 3672,  "SMILES": "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "MW": 206.29},
        "caffeine":    {"CID": 2519,  "SMILES": "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "MW": 194.19},
        "paracetamol": {"CID": 1983,  "SMILES": "CC(=O)Nc1ccc(O)cc1", "MW": 151.16},
        "sildenafil":  {"CID": 135398744, "SMILES": "CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1", "MW": 474.58},
    }

    result = DB.get(name.lower().strip())
    return json.dumps(result if result else {"error": f"Not found: {name}"})

# Register tools
TOOLS = {
    "admet":      Tool("admet",     "Compute ADMET descriptors (MW, LogP, TPSA, HBD, HBA, QED, Ro5 violations)", tool_admet),
    "herg":       Tool("herg",      "Predict hERG cardiotoxicity risk and estimated IC50", tool_herg),
    "pains":      Tool("pains",     "Screen for PAINS (pan-assay interference) structural alerts", tool_pains),
    "similarity": Tool("similarity","Compute Tanimoto similarity (ECFP4) between two SMILES", tool_similarity),
    "scaffold":   Tool("scaffold",  "Extract Murcko scaffold and ring analysis", tool_scaffold),
    "pubchem":    Tool("pubchem",   "Look up a compound by name in PubChem to get SMILES and properties", tool_lookup_pubchem),
}

TOOL_DESCRIPTIONS = "\n".join(f"  {name}: {t.description}" for name, t in TOOLS.items())
print("Tool registry created:")
for name, tool in TOOLS.items():
    print(f"  ✓ {name}")

Tool registry created:
  ✓ admet
  ✓ herg
  ✓ pains
  ✓ similarity
  ✓ scaffold
  ✓ pubchem


---
## Section 2 — ReAct Agent

**ReAct** (Yao 2022): interleave **Reasoning** and **Acting**.

```
Thought: I need to assess cardiac safety of this compound.
Action: herg
Input: CC(=O)Oc1ccccc1C(=O)O
Observation: {"hERG_risk": "LOW", ...}
Thought: hERG is low risk. Now check drug-likeness.
Action: admet
...
Final Answer: The compound is safe and drug-like.
```

In [12]:
# ── 2.1 ReAct agent — pure Python (no LangChain required) ───────────────────
# This works with any LLM API. We simulate the LLM for reproducibility.

class SimulatedLLM:
    """
    Simulated LLM for tutorial purposes.
    Replace with: openai.ChatCompletion.create() or langchain ChatOpenAI
    """
    def __init__(self):
        self.call_count = 0

    def __call__(self, prompt: str) -> str:
        self.call_count += 1
        # Simple rule-based simulation of ReAct reasoning
        if "assess" in prompt.lower() and "Thought:" not in prompt:
            return "Thought: I should first check ADMET properties to assess drug-likeness.\nAction: admet\nInput: {smiles}"
        elif "Observation:" in prompt and "hERG" not in prompt:
            return "Thought: Drug-likeness assessed. Now I must check cardiac safety.\nAction: herg\nInput: {smiles}"
        elif "hERG" in prompt and "pains" not in prompt.lower():
            return "Thought: Cardiac safety checked. Let me screen for PAINS interference.\nAction: pains\nInput: {smiles}"
        else:
            return "Thought: I have enough information.\nFinal Answer: Assessment complete. See observations above."

llm = SimulatedLLM()

def react_agent(query: str, smiles: str, max_steps: int = 8) -> str:
    """
    ReAct agent loop.

    Args:
        query:     Natural language request (e.g. 'assess this compound')
        smiles:    SMILES string of the compound
        max_steps: Maximum number of Thought/Action/Observation cycles

    Returns:
        Final answer string
    """
    # Build the system prompt
    system = f"""You are a computational toxicologist. You have these tools:
{TOOL_DESCRIPTIONS}

Use this format EXACTLY:
Thought: <reasoning about what to do next>
Action: <tool_name>
Input: <smiles_or_argument>
Observation: <tool_result — filled in automatically>
... (repeat as needed)
Thought: I have enough information.
Final Answer: <your conclusion>
"""

    trajectory = []
    prompt = f"{system}\n\nQuery: {query}\nSMILES: {smiles}\n\n"

    for step in range(max_steps):
        # LLM generates next thought + action
        response = llm(prompt).replace("{smiles}", smiles)
        trajectory.append(f"Step {step+1}: {response}")

        if "Final Answer:" in response:
            break

        # Parse action
        action_match = re.search(r"Action:\s*(\w+)", response)
        input_match  = re.search(r"Input:\s*(.+?)(?:\n|$)", response)

        if not action_match:
            break

        tool_name = action_match.group(1).strip()
        tool_input = input_match.group(1).strip() if input_match else smiles

        # Execute tool
        if tool_name in TOOLS:
            if tool_name == "similarity":
                observation = "Need two SMILES for similarity — skipping."
            else:
                observation = TOOLS[tool_name].run(smiles=tool_input)
        else:
            observation = f"Unknown tool: {tool_name}"

        prompt += f"{response}\nObservation: {observation}\n\n"
        trajectory.append(f"  → {tool_name}({tool_input[:40]}) = {observation[:80]}...")

    return "\n".join(trajectory)

# Run the agent
print("=" * 60)
print("ReAct Agent — Aspirin Safety Assessment")
print("=" * 60)
result = react_agent(
    query="Assess the cardiac safety and drug-likeness of this compound.",
    smiles="CC(=O)Oc1ccccc1C(=O)O"   # Aspirin
)
print(result)

ReAct Agent — Aspirin Safety Assessment
Step 1: Thought: I have enough information.
Final Answer: Assessment complete. See observations above.


---
## Section 3 — Reflexion: Self-Critiquing Agent

**Shinn 2023**: the agent generates a report, critiques its own quality, then revises.

```
Generate → Critique (score 0–1) → Revise → Critique → ... → Final
             ↑_________________________|
         Repeat if score < threshold
```

In [14]:
# ── 3.1 Reflexion agent ──────────────────────────────────────────────────────
def critique_report(report: str, smiles: str) -> tuple[float, list[str]]:
    """
    Evaluate report quality. Returns (score 0-1, list of gaps).
    In production: use an LLM as the critic.
    """
    mol = Chem.MolFromSmiles(smiles)
    gaps = []
    score = 1.0

    checklist = {
        "MW":     ("molecular weight" in report.lower() or "mw" in report.lower()),
        "hERG":   ("herg" in report.lower() or "cardiac" in report.lower()),
        "PAINS":  ("pains" in report.lower() or "interference" in report.lower()),
        "Ro5":    ("ro5" in report.lower() or "lipinski" in report.lower() or "oral" in report.lower()),
        "Conclusion": ("conclusion" in report.lower() or "recommend" in report.lower() or "suitable" in report.lower()),
    }

    for item, present in checklist.items():
        if not present:
            gaps.append(f"Missing: {item}")
            score -= 0.2

    return max(0, score), gaps

def reflexion_agent(smiles: str, max_iterations: int = 3,
                    quality_threshold: float = 0.8) -> dict:
    """
    Reflexion agent:
      1. Generate initial safety report
      2. Critique the report
      3. Revise if quality below threshold
      4. Repeat up to max_iterations
    """
    mol = _mol(smiles)
    history = []

    # Gather all tool outputs first
    admet_data  = json.loads(TOOLS["admet"].run(smiles=smiles))
    herg_data   = json.loads(TOOLS["herg"].run(smiles=smiles))
    pains_data  = json.loads(TOOLS["pains"].run(smiles=smiles))
    scaffold_d  = json.loads(TOOLS["scaffold"].run(smiles=smiles))

    def generate_report(iteration: int, prev_gaps: list = None) -> str:
        """Generate (or revise) a safety report."""
        base = f"""Safety Assessment Report — Iteration {iteration}
    SMILES: {smiles}

    Molecular Properties:
      MW = {admet_data['MW']} Da  |  LogP = {admet_data['LogP']}  |  TPSA = {admet_data['TPSA']} Å²
      HBD = {admet_data['HBD']}  |  HBA = {admet_data['HBA']}  |  QED = {admet_data['QED']}
      Ro5 violations: {admet_data['Ro5_violations']} → oral bioavailability: {admet_data['oral_bioavailability']}

    Cardiac Safety (hERG):
      Risk level: {herg_data['hERG_risk']}
      Estimated IC50: {herg_data['estimated_IC50']}
      {herg_data.get('warning', '')}

    PAINS screening:
      Alert detected: {pains_data['pains_alert']}
      {pains_data.get('pattern', '')}

    Scaffold: {scaffold_d['scaffold_smiles']}  ({scaffold_d['ring_count']} rings)
    """
        # Add conclusion on later iterations
        if iteration > 1 or (prev_gaps and "Conclusion" in str(prev_gaps)):
            overall = "NOT SUITABLE" if (herg_data['hERG_risk']=="HIGH" or
                       pains_data['pains_alert'] or admet_data['Ro5_violations']>1) else "SUITABLE"
            base += f"\nConclusion: Compound is {overall} for advancement.\n"
            base += "Recommendation: " + (
                "Proceed to in vitro assay" if overall=="SUITABLE"
                else "Requires structural modification to address flagged issues")
        return base

    report = generate_report(iteration=1)
    for iteration in range(1, max_iterations + 1):
        score, gaps = critique_report(report, smiles)
        history.append({"iteration": iteration, "score": score, "gaps": gaps})

        print(f"  Iteration {iteration}: quality = {score:.1f}  gaps = {gaps}")

        if score >= quality_threshold:
            print(f"  → Quality threshold reached at iteration {iteration}")
            break

        if iteration < max_iterations:
            report = generate_report(iteration=iteration+1, prev_gaps=gaps)

    return {"final_report": report, "history": history, "final_score": score}

print("Reflexion Agent — Compound Assessment")
print("=" * 60)
result = reflexion_agent("CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1", max_iterations=3)
print("\nFinal Report:")
print(result["final_report"])
print(f"\nIteration history: {[(h['iteration'], h['score']) for h in result['history']]}")

Reflexion Agent — Compound Assessment
  Iteration 1: quality = 0.8  gaps = ['Missing: Conclusion']
  → Quality threshold reached at iteration 1

Final Report:
Safety Assessment Report — Iteration 1
    SMILES: CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1

    Molecular Properties:
      MW = 304.5 Da  |  LogP = 3.29  |  TPSA = 15.7 Å²
      HBD = 0  |  HBA = 3  |  QED = 0.733
      Ro5 violations: 0 → oral bioavailability: Likely

    Cardiac Safety (hERG):
      Risk level: LOW
      Estimated IC50: > 10 μM (likely safe)
      

    PAINS screening:
      Alert detected: False
      None

    Scaffold: c1ccc(CC2CCNCC2)cc1  (2 rings)
    

Iteration history: [(1, 0.8)]


---
## Section 4 — RAG Agent

Retrieval-Augmented Generation: ground every claim in a real source.

```
Query → Retrieve relevant documents → LLM answers using ONLY retrieved context
                                       → Citations in answer
```

In [ ]:
# ── 4.1 Build a toxicology knowledge base ────────────────────────────────────
# TF-IDF retrieval over curated toxicology literature

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

KNOWLEDGE_BASE = [
    {
        "source": "ICH S7B (2005)", "year": 2005,
        "text": "The hERG potassium channel is the primary molecular target for drug-induced QT prolongation. ICH S7B recommends in vitro hERG assay and in vivo QT assessment for all new chemical entities. An IC50 > 30× therapeutic Cmax is generally considered acceptable."
    },
    {
        "source": "Lipinski et al. (1997)", "year": 1997,
        "text": "The Rule of Five (Ro5) predicts oral bioavailability: MW ≤ 500, LogP ≤ 5, HBD ≤ 5, HBA ≤ 10. Compounds violating more than one rule typically have poor absorption and permeability."
    },
    {
        "source": "Baell & Holloway (2010)", "year": 2010,
        "text": "PAINS (pan-assay interference compounds) produce false positives across multiple biochemical assays due to non-specific reactivity. Rhodanines, catechols, quinones, and Michael acceptors are common PAINS scaffolds. PAINS filtering should be applied before HTS campaigns."
    },
    {
        "source": "DILIrank Database (Chen 2016)", "year": 2016,
        "text": "DILIrank classifies 1036 drugs by DILI severity: most-concern, less-concern, no-concern, and ambiguous. High DILI risk correlates with mitochondrial liability, bile salt export pump (BSEP) inhibition, and reactive metabolite formation."
    },
    {
        "source": "Veber et al. (2002)", "year": 2002,
        "text": "Oral bioavailability in rats is predicted by: rotatable bonds ≤ 10 and TPSA ≤ 140 Å² (or polar atoms ≤ 12). These Veber rules complement the Lipinski Ro5 for improved bioavailability prediction."
    },
    {
        "source": "Ames (1973)", "year": 1973,
        "text": "The Ames test detects mutagenic potential using Salmonella bacteria. ICH M7 classifies mutagens into 5 classes. Structural alerts (SMARTS) for DNA-reactive compounds include nitroaromatics, alkylating agents, and Michael acceptors. Class 1 and 2 impurities must be controlled to TTC of 1.5 μg/day."
    },
    {
        "source": "Pfizer CNS MPO (Wager 2010)", "year": 2010,
        "text": "The CNS multiparameter optimisation (MPO) score uses six physicochemical properties: MW ≤ 360, LogP 1–3, TPSA ≤ 90, HBD 0–1, LogP < 5, aromatic rings ≤ 2. CNS drugs typically score 4–6 out of 6. Higher MPO correlates with better CNS penetration, safety, and clinical success."
    },
    {
        "source": "CiPA Initiative (2016)", "year": 2016,
        "text": "The Comprehensive In vitro Proarrhythmia Assay (CiPA) paradigm replaces ICH S7B hERG-only testing. CiPA integrates multi-channel ion current assays (hERG, Nav1.5, Cav1.2, IKs), in silico AP modelling, and stem cell-derived cardiomyocyte assays to predict TdP risk more accurately."
    },
]

# Build TF-IDF index
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
kb_texts = [doc["text"] for doc in KNOWLEDGE_BASE]
kb_matrix = vectorizer.fit_transform(kb_texts)

def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Retrieve top-k most relevant documents for a query."""    query_vec = vectorizer.transform([query])
    scores    = cosine_similarity(query_vec, kb_matrix).flatten()
    top_idx   = scores.argsort()[::-1][:top_k]
    return [
        {**KNOWLEDGE_BASE[i], "relevance_score": round(float(scores[i]), 3)}
        for i in top_idx if scores[i] > 0.05
    ]

def rag_agent(smiles: str, question: str) -> dict:
    """
    RAG-grounded safety assessment.
    Every statement is backed by a retrieved source.
    """
    # Gather tool data
    admet_data = json.loads(TOOLS["admet"].run(smiles=smiles))
    herg_data  = json.loads(TOOLS["herg"].run(smiles=smiles))
    pains_data = json.loads(TOOLS["pains"].run(smiles=smiles))

    # Build query from question + compound properties
    augmented_query = f"{question} hERG {herg_data['hERG_risk']} ADMET LogP {admet_data['LogP']}"
    docs = retrieve(augmented_query, top_k=4)

    # Build cited answer
    findings = []
    citations = []
    for doc in docs:
        findings.append(f"  • [{doc['source']}]: {doc['text'][:120]}...")
        citations.append(f"{doc['source']} ({doc['year']})")

    answer = f"""RAG Safety Assessment
SMILES: {smiles}
Question: {question}

Compound Data:
  MW={admet_data['MW']} Da | LogP={admet_data['LogP']} | TPSA={admet_data['TPSA']} Å²
  Ro5 violations: {admet_data['Ro5_violations']} | hERG risk: {herg_data['hERG_risk']}
  PAINS: {'DETECTED — ' + pains_data.get('pattern','') if pains_data['pains_alert'] else 'Clean'}

Retrieved Evidence:
{chr(10).join(findings)}

Cited Answer:
  Based on {citations[0] if citations else 'available data'}, this compound shows
  {'CONCERN' if herg_data['hERG_risk']=='HIGH' else 'LOW CONCERN'} for cardiac safety.
  Oral bioavailability predicted to be {admet_data['oral_bioavailability'].lower()} per {citations[1] if len(citations)>1 else 'Ro5'}.

References: {'; '.join(citations)}
"""
    return {"answer": answer, "retrieved_docs": docs}

print("RAG Agent:")
result = rag_agent("CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1",
                   "Is this compound safe from a cardiac and regulatory standpoint?")
print(result["answer"])

---
## Section 5 — Multi-Agent System

Specialist agents + a supervisor that routes queries and assembles the final answer.

In [ ]:
# ── 5.1 Specialist agents ─────────────────────────────────────────────────────
class SpecialistAgent:
    """A focused agent with a specific expertise area."""    def __init__(self, name: str, tools: list[str], role: str):
        self.name  = name
        self.tools = tools
        self.role  = role

    def run(self, smiles: str, task: str) -> dict:
        results = {}
        for tool_name in self.tools:
            if tool_name in TOOLS:
                raw = TOOLS[tool_name].run(smiles=smiles)
                try:
                    results[tool_name] = json.loads(raw)
                except json.JSONDecodeError:
                    results[tool_name] = raw
        return {"agent": self.name, "task": task, "results": results}

# Create specialists
AGENTS = {
    "admet_agent":   SpecialistAgent("ADMET Specialist",
                        ["admet"], "Assess drug-likeness and ADMET profile"),
    "cardiac_agent": SpecialistAgent("Cardiac Safety Specialist",
                        ["herg"], "Evaluate cardiac safety and hERG liability"),
    "tox_agent":     SpecialistAgent("Toxicology Specialist",
                        ["pains"], "Screen for toxic structural features and interference"),
    "chem_agent":    SpecialistAgent("Cheminformatics Specialist",
                        ["scaffold", "similarity"], "Analyse chemical structure and scaffold"),
    "lit_agent":     SpecialistAgent("Literature Specialist",
                        ["pubchem"], "Retrieve literature data and known compound information"),
}

class SupervisorAgent:
    """Coordinates specialist agents and assembles the final report."""    def __init__(self, agents: dict, kb_retriever):
        self.agents    = agents
        self.retriever = kb_retriever

    def route(self, question: str) -> list[str]:
        """Decide which specialists to call based on the question."""        q = question.lower()
        needed = []
        if any(w in q for w in ["safety", "cardiac", "herg", "qt"]): needed.append("cardiac_agent")
        if any(w in q for w in ["admet", "bioavail", "absorption", "druglike"]): needed.append("admet_agent")
        if any(w in q for w in ["tox", "pains", "alert", "genotox"]): needed.append("tox_agent")
        if any(w in q for w in ["scaffold", "structure", "similar"]): needed.append("chem_agent")
        if any(w in q for w in ["literature", "known", "pubchem", "database"]): needed.append("lit_agent")
        # Default: run all if nothing matched
        return needed or list(self.agents.keys())

    def run(self, smiles: str, question: str) -> str:
        print(f"  Supervisor routing '{question[:50]}...'")
        selected = self.route(question)
        print(f"  → Dispatching to: {selected}")

        outputs = {}
        for agent_name in selected:
            print(f"    Running {agent_name}...")
            outputs[agent_name] = self.agents[agent_name].run(smiles, question)

        # Retrieve supporting literature
        docs = self.retriever(question, top_k=2)

        # Assemble final answer
        report_lines = [f"Multi-Agent Safety Report", f"SMILES: {smiles}", f"Question: {question}", ""]
        for agent_name, output in outputs.items():
            report_lines.append(f"[{output['agent']}]")
            for tool, result in output["results"].items():
                if isinstance(result, dict):
                    for k, v in list(result.items())[:3]:
                        report_lines.append(f"  {k}: {v}")
            report_lines.append("")

        report_lines.append("Supporting Literature:")
        for doc in docs:
            report_lines.append(f"  [{doc['source']}]: {doc['text'][:100]}...")

        return "\n".join(report_lines)

supervisor = SupervisorAgent(AGENTS, retrieve)
print("Multi-Agent System")
print("=" * 60)
report = supervisor.run(
    "CCCC1=NN(C)C(=O)c2cc(cnc21)c1ccc(cc1)S(=O)(=O)N1CCN(C)CC1",  # Sildenafil-like
    "Comprehensive safety and drug-likeness assessment for cardiac and ADMET concerns"
)
print(report)

---
## Section 6 — LangGraph State Machine

LangGraph represents an agent as a **directed graph** where each node is a function and edges are conditional.

In [ ]:
# ── 6.1 LangGraph-style state machine (pure Python) ──────────────────────────
# We implement the core pattern without requiring the full LangGraph install.

from typing import TypedDict, Literal
from enum import Enum

class RiskLevel(str, Enum):
    LOW    = "LOW"
    MEDIUM = "MEDIUM"
    HIGH   = "HIGH"

class CompoundState(TypedDict):
    """Typed state passed between nodes in the graph."""    smiles:          str
    step:            str
    admet:           dict
    herg_risk:       str
    pains_alert:     bool
    risk_level:      str
    requires_hitl:   bool
    hitl_approved:   bool | None
    lead_opt:        dict
    final_report:    str
    messages:        list[str]

def node_validate(state: CompoundState) -> CompoundState:
    """Node 1: validate SMILES."""    mol = Chem.MolFromSmiles(state["smiles"])
    state["step"] = "validated"
    state["messages"].append(f"Validate: {'OK' if mol else 'INVALID'}")
    return state

def node_admet(state: CompoundState) -> CompoundState:
    """Node 2: compute ADMET."""    data = json.loads(TOOLS["admet"].run(smiles=state["smiles"]))
    state["admet"]   = data
    state["step"]    = "admet_done"
    state["messages"].append(f"ADMET: QED={data['QED']}, Ro5_viol={data['Ro5_violations']}")
    return state

def node_cardiac(state: CompoundState) -> CompoundState:
    """Node 3: hERG safety."""    data = json.loads(TOOLS["herg"].run(smiles=state["smiles"]))
    state["herg_risk"] = data["hERG_risk"]
    state["step"]      = "cardiac_done"
    state["messages"].append(f"hERG: {data['hERG_risk']} (score={data['risk_score']})")
    return state

def node_tox(state: CompoundState) -> CompoundState:
    """Node 4: PAINS screen."""    data = json.loads(TOOLS["pains"].run(smiles=state["smiles"]))
    state["pains_alert"] = data["pains_alert"]
    state["step"]        = "tox_done"
    state["messages"].append(f"PAINS: {'ALERT' if data['pains_alert'] else 'clean'}")
    return state

def node_risk_tier(state: CompoundState) -> CompoundState:
    """Node 5: compute overall risk tier."""    h = state.get("herg_risk", "LOW")
    p = state.get("pains_alert", False)
    v = state.get("admet", {}).get("Ro5_violations", 0)

    if h == "HIGH" or p:
        risk = "HIGH"
    elif h == "MEDIUM" or v >= 2:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    state["risk_level"]    = risk
    state["requires_hitl"] = (risk == "HIGH")
    state["step"]          = "risk_tiered"
    state["messages"].append(f"Risk tier: {risk} (requires_hitl={risk=='HIGH'})")
    return state

def node_hitl(state: CompoundState) -> CompoundState:
    """Node 6: human-in-the-loop review (auto-approve in tutorial)."""    # In production: pause, send alert, wait for human decision
    print(f"  [HITL] HIGH-RISK compound flagged for human review!")
    print(f"  [HITL] Auto-approving for tutorial (would block in production)")
    state["hitl_approved"] = False   # reject high-risk in demo
    state["step"]          = "hitl_done"
    state["messages"].append("HITL: Reviewed and REJECTED (too risky)")
    return state

def node_lead_opt(state: CompoundState) -> CompoundState:
    """Node 7: generate lead optimisation suggestions."""    suggestions = []
    admet = state.get("admet", {})

    if admet.get("LogP", 0) > 4:
        suggestions.append("Reduce LogP: add polar groups, replace aryl with heteroaryl")
    if admet.get("HBD", 0) > 3:
        suggestions.append("Reduce HBD: cap free NH/OH with methyl or acyl group")
    if state.get("herg_risk") in ("MEDIUM", "HIGH"):
        suggestions.append("Reduce hERG: break up aryl rings, reduce lipophilicity, reduce basic N")
    if not suggestions:
        suggestions.append("Compound within acceptable parameters — proceed to synthesis")

    state["lead_opt"] = {"suggestions": suggestions}
    state["step"]     = "lead_opt_done"
    state["messages"].append(f"Lead opt: {len(suggestions)} suggestions")
    return state

def node_report(state: CompoundState) -> CompoundState:
    """Node 8: generate final report."""    admet = state.get("admet", {})
    status = "REJECTED" if state.get("hitl_approved") == False else "APPROVED"
    state["final_report"] = f"""
Final Report | {status}
SMILES: {state['smiles']}
Risk level: {state.get('risk_level', 'N/A')}
QED: {admet.get('QED','N/A')} | LogP: {admet.get('LogP','N/A')} | Ro5: {admet.get('Ro5_violations','N/A')} violations
hERG: {state.get('herg_risk','N/A')} | PAINS: {'YES' if state.get('pains_alert') else 'No'}
Lead opt: {'; '.join(state.get('lead_opt',{}).get('suggestions',['N/A'])[:2])}
Trajectory: {' → '.join(state['messages'])}
"""
    state["step"] = "done"
    return state

# ── Conditional edge function ──────────────────────────────────────────────────
def route_after_risk(state: CompoundState) -> Literal["hitl", "lead_opt"]:
    """If HIGH risk → human review. Otherwise → lead opt."""    return "hitl" if state.get("requires_hitl") else "lead_opt"

def run_graph(smiles: str) -> CompoundState:
    """Run the agent state machine."""    state: CompoundState = {
        "smiles": smiles, "step": "start", "admet": {}, "herg_risk": "",
        "pains_alert": False, "risk_level": "", "requires_hitl": False,
        "hitl_approved": None, "lead_opt": {}, "final_report": "", "messages": []
    }

    # Execute nodes in order (in LangGraph, this is the graph topology)
    nodes = [node_validate, node_admet, node_cardiac, node_tox, node_risk_tier]
    for node in nodes:
        state = node(state)

    # Conditional routing
    next_node = route_after_risk(state)
    print(f"  Conditional edge → {next_node}")
    if next_node == "hitl":
        state = node_hitl(state)
    state = node_lead_opt(state)
    state = node_report(state)
    return state

# Run on two compounds
for smi, label in [
    ("CC(=O)Oc1ccccc1C(=O)O", "Aspirin (low risk)"),
    ("CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1", "hERG-active compound"),
]:
    print(f"\n{'='*60}")
    print(f"Running graph for: {label}")
    result = run_graph(smi)
    print(result["final_report"])

---
## Section 7 — Memory Systems

In [ ]:
# ── 7.1 Episodic + semantic memory ───────────────────────────────────────────
from collections import deque

class AgentMemory:
    """
    Two memory types:
      Episodic  — what happened in past interactions (bounded queue)
      Semantic  — distilled facts that persist (dictionary)
    """
    def __init__(self, max_episodes: int = 50):
        self.episodic  = deque(maxlen=max_episodes)
        self.semantic  = {}

    def add_episode(self, smiles: str, action: str, result: str, outcome: str):
        """Record one agent interaction."""        self.episodic.append({
            "smiles": smiles, "action": action,
            "result": result[:80], "outcome": outcome,
            "timestamp": time.time()
        })

    def remember_fact(self, key: str, value):
        """Store a distilled fact."""        self.semantic[key] = value

    def recall_similar(self, smiles: str, n: int = 3) -> list[dict]:
        """Find similar past episodes by Tanimoto similarity."""        from rdkit import DataStructs
        query_mol = Chem.MolFromSmiles(smiles)
        if query_mol is None:
            return []
        query_fp = AllChem.GetMorganFingerprintAsBitVect(query_mol, 2, 2048)

        ranked = []
        for ep in self.episodic:
            mol = Chem.MolFromSmiles(ep["smiles"])
            if mol:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
                from rdkit import DataStructs as DS
                tc = DS.TanimotoSimilarity(query_fp, fp)
                ranked.append((tc, ep))

        ranked.sort(key=lambda x: x[0], reverse=True)
        return [ep for _, ep in ranked[:n]]

    def summary(self) -> str:
        return (f"Memory: {len(self.episodic)} episodes, "
                f"{len(self.semantic)} semantic facts")

# Demo
memory = AgentMemory()

# Populate with past assessments
past = [
    ("CC(=O)Oc1ccccc1C(=O)O",    "admet",  "QED=0.55",  "APPROVED"),
    ("CC(=O)Nc1ccc(O)cc1",        "herg",   "LOW risk",  "APPROVED"),
    ("CCN(CC)CCOc1ccc(CC2CCN(C)CC2)cc1", "herg", "HIGH risk", "REJECTED"),
]
for smi, action, result, outcome in past:
    memory.add_episode(smi, action, result, outcome)

memory.remember_fact("hERG_HIGH_threshold", 1.0)
memory.remember_fact("preferred_clogp_range", (1, 3))

# Query memory for a new compound
new_smi = "CCN(CC)CCc1ccc(OC)cc1"  # similar to the hERG-active compound
similar = memory.recall_similar(new_smi, n=2)

print(memory.summary())
print(f"\nSimilar past cases for new compound:")
for ep in similar:
    print(f"  {ep['smiles'][:40]:40s} → {ep['outcome']}  ({ep['result']})")
print(f"\nSemantic facts: {memory.semantic}")

---
## Section 8 — Best Practices & Production Patterns

In [ ]:
# ── 8.1 Cheatsheet ───────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║            Agentic AI for Drug Discovery — Quick Reference              ║
╠══════════════════════════════════════════════════════════════════════════╣
║ PATTERNS                                                                 ║
║  ReAct        Thought → Action → Observation (loop)   Yao 2022         ║
║  Reflexion    Generate → Critique → Revise             Shinn 2023       ║
║  RAG          Retrieve → Ground → Cite                 Lewis 2020       ║
║  Multi-agent  Supervisor + Specialists                 ChatInvent AZ    ║
║  LangGraph    StateGraph + conditional edges           LangChain Inc    ║
║  HITL         Human-in-loop for high-risk decisions                     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ TOOL DESIGN                                                              ║
║  ✓ Each tool = one pure function with clear in/out types                ║
║  ✓ Tools return strings (JSON.dumps) — easy for LLM to parse           ║
║  ✓ Always handle errors inside tool.run() — never let agent crash       ║
║  ✓ Include description: what the tool does AND when to use it           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ MEMORY                                                                   ║
║  Episodic     deque(maxlen=N)  — recent interaction history             ║
║  Semantic     dict             — distilled facts & rules                ║
║  Similarity   Tanimoto recall  — find analogous past cases             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ TOXICOLOGY TOOLS TO IMPLEMENT                                            ║
║  admet         → MW, LogP, TPSA, HBD, HBA, QED, Ro5                   ║
║  herg          → IC50 prediction, CiPA risk tier                        ║
║  pains         → FilterCatalog.PAINS                                   ║
║  ames          → ICH M7 SMARTS alert screening                         ║
║  dili          → DILIrank ML predictor                                  ║
║  similarity    → Tanimoto vs reference set                             ║
║  scaffold      → Murcko decomposition                                  ║
║  pubchem       → REST API / RDKit PandasTools                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║ PITFALLS                                                                 ║
║  ✗ No tool error handling → agent crashes on bad SMILES                ║
║  ✗ Too many tools → LLM confused, wrong tool called                    ║
║  ✗ No HITL for HIGH risk → dangerous recommendations                   ║
║  ✗ No memory → repeats same mistakes, wastes LLM calls                 ║
║  ✗ Non-deterministic routing → hard to debug                           ║
╚══════════════════════════════════════════════════════════════════════════╝
""")